## Convert a notebook to PDF

Upload the `.ipynb` when prompted; the PDF downloads automatically.

Note: images referenced by a **relative** path (`img/...`) will not appear, because only the
notebook itself is uploaded. Either use full `https://` image links, or upload the `img` folder
to the Colab session first.

In [ ]:
!pip install nbconvert weasyprint -q

In [ ]:
from weasyprint import HTML, CSS
from nbconvert import HTMLExporter
import nbformat
from google.colab import files

uploaded = files.upload()
notebook_filename = list(uploaded.keys())[0]
print(f"Uploaded notebook: {notebook_filename}")

html_filename = notebook_filename.replace('.ipynb', '.html')
pdf_filename  = notebook_filename.replace('.ipynb', '.pdf')

with open(notebook_filename, 'r', encoding='utf-8') as f:
    notebook_node = nbformat.read(f, as_version=4)

# hide the In[ ]/Out[ ] prompts - they push the content off the page
html_exporter = HTMLExporter(exclude_input_prompt=True, exclude_output_prompt=True)
(body, resources) = html_exporter.from_notebook_node(notebook_node)

with open(html_filename, "w", encoding="utf-8") as f:
    f.write(body)

# print stylesheet: fit the notebook to A4 instead of letting it overflow the page edge
print_css = CSS(string="""
@page { size: A4 portrait; margin: 14mm 12mm; }
html, body { font-size: 11px !important; }
.jp-Notebook, .jp-Cell, .jp-Cell-inputWrapper, .jp-Cell-outputWrapper,
.jp-InputArea, .jp-OutputArea, .jp-RenderedHTMLCommon, .jp-InputArea-editor {
  max-width: 100% !important; width: auto !important;
  padding-left: 0 !important; padding-right: 0 !important;
  overflow: visible !important; overflow-wrap: break-word !important;
}
.jp-InputPrompt, .jp-OutputPrompt,
.jp-Cell-inputCollapser, .jp-Cell-outputCollapser { display: none !important; }
img { max-width: 100% !important; height: auto !important; }
pre, code, .highlight pre { white-space: pre-wrap !important; word-break: break-word !important; }
table { max-width: 100% !important; table-layout: fixed !important; }
h1 { font-size: 1.9em !important; } h2 { font-size: 1.5em !important; }
""")

HTML(html_filename).write_pdf(pdf_filename, stylesheets=[print_css])
files.download(pdf_filename)